# Rung 40 — EVAL: score an arm, read it against rung 38

Written **before** the arms ran, on purpose. Rung 38's eval cell had no code when its arm
finished, so the eval was written that same night against a checkpoint that already existed.
Rung 40 has **two** arms; the same gap would cost twice.

🔴 **RULES §EVAL is binding.** Score ONLY through `frame.metrics`; leaf→group via `Capability.group`;
ID/OOD from the **qID prefix**; the paired CI is **video-clustered** — effective n is ~38 videos,
not 6252 questions, and an unclustered CI would be ~10× too narrow.

⚠️ **Control = rung 38's OWN arm** (`38_qwen36_27b_v1` ep1), **not A2** — A2 is a different
backbone and using it would compare two variables labelled as one.

Declared primary cell: **`proxy_leaderboard`**. Everything else is exploratory and is reported
beside it, never quoted as the result.


## 1 — Parameters (papermill overrides)

In [ ]:
MERGED_DIR = ""      # the arm's merged/ -- set this
RUN_NAME   = "40_A_alpha_v1"
SMOKE      = True    # True -> n_eval=40, for a path check


## 2 — Config + the four asserts that keep the inference path from moving

In [ ]:
import logging, sys
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
sys.path.insert(0, str(Path.cwd() / "_tools"))

from eval_arm import EvalConfig, score, paired_vs_control, verdict, write, CONTROL  # noqa: E402

cfg = EvalConfig(
    merged_dir=MERGED_DIR,
    out_dir=str(Path.cwd() / "runs" / RUN_NAME / "eval"),
    run_name=RUN_NAME,
    smoke=SMOKE,
)
cfg


## 3 — Score. Canonical path only.

In [ ]:
report = score(cfg)

## 4 — Paired, video-clustered CI against rung 38

Skipped in SMOKE: 40 questions cannot support a clustered CI, and reporting one would be worse
than reporting none.

In [ ]:
paired = {}
if not SMOKE:
    arm_csv = str(Path(cfg.out_dir) / "results.csv")
    paired = paired_vs_control(arm_csv, cfg)
    for cell, d in paired.items():
        print(f"  {cell:<5} delta={d['delta']:+.4f}  CI=[{d['ci_low']:+.4f},{d['ci_high']:+.4f}]  "
              f"n={d['n']} videos={d['n_videos']}")
else:
    print("SMOKE: paired CI skipped -- 40 questions cannot support a clustered CI")


## 5 — Verdict against the pre-registered condition. No post-hoc re-cutting.

In [ ]:
v = verdict(report, paired)
for k, val in v.items():
    print(f"  {k:<20} {val}")

payload = {"report": report, "paired": paired, "verdict": v, "control": CONTROL}
print()
print(write(cfg, payload))
